In [4]:
import pandas as pd
from datetime import datetime, timezone

In [25]:
kalshi_df = pd.read_csv("/Users/marcotaylhardat/Documents/ML/kalshi_full_data.csv")
markets_df = pd.read_csv("/Users/marcotaylhardat/Documents/ML/kalshi_markets.csv")

markets_df_cols = markets_df[["close_time", "open_time", "ticker"]]

In [ ]:
## Manipulate datetime to get closest ticker to timestamp
kalshi_df['ticker_suffix'] = kalshi_df['ticker'].str.split('-').str[-1]
kalshi_df['ticker_middle'] = kalshi_df['ticker'].str.split('-').str[1]

kalshi_w_markets = pd.merge(kalshi_df, markets_df_cols, left_on='ticker', right_on='ticker', how='left')

kalshi_w_markets['close_time'] = pd.to_datetime(kalshi_w_markets['close_time'], utc=True)
kalshi_w_markets['open_time'] = pd.to_datetime(kalshi_w_markets['open_time'], utc=True)
kalshi_w_markets['timestamp'] = pd.to_datetime(kalshi_w_markets['timestamp'], utc=True)
kalshi_w_markets['time_to_close'] = (kalshi_w_markets['close_time'] - kalshi_w_markets['timestamp']).dt.total_seconds()

# rank within each timestamp group so the smallest time_to_close gets rank 1 per timestamp
kalshi_w_markets['close_rank'] = kalshi_w_markets.groupby('timestamp')['time_to_close']\
    .rank(method='dense', ascending=True)

kalshi_w_markets = kalshi_w_markets[kalshi_w_markets['close_rank'] == 1]

#kalshi_w_markets.to_csv("/Users/marcotaylhardat/Documents/ML/time_to_close_test.csv", index=False)

In [55]:
kalshi_df_wide = kalshi_df.pivot_table(index=["timestamp"], columns="ticker_suffix", values=["volume", "open", "high", "low", "close", "mean"]).reset_index()

print(kalshi_df_wide.head())

                         timestamp close                        high        \
ticker_suffix                        C25   C26    H0  H25  H26   C25   C26   
0              2025-07-30 15:00:00  61.0   NaN  35.0  NaN  NaN  70.0   NaN   
1              2025-07-30 16:00:00  60.0  10.0  38.0  3.0  1.0  65.0  10.0   
2              2025-07-30 17:00:00  65.0   NaN  37.0  NaN  NaN  65.0   NaN   
3              2025-07-30 18:00:00   NaN   NaN   NaN  NaN  NaN   NaN   NaN   
4              2025-07-30 19:00:00  53.0   4.0  46.0  NaN  NaN  65.0   7.0   

                          ...  open                        volume          \
ticker_suffix    H0  H25  ...   C25  C26    H0  H25  H26      C25     C26   
0              35.0  NaN  ...  70.0  NaN  25.0  NaN  NaN     76.0     0.0   
1              38.0  3.0  ...  60.0  9.0  35.0  3.0  1.0    286.0   100.0   
2              37.0  NaN  ...  65.0  NaN  34.0  NaN  NaN      7.0     0.0   
3               NaN  NaN  ...   NaN  NaN   NaN  NaN  NaN      0.0   

In [56]:
kalshi_df_wide.to_csv("/Users/marcotaylhardat/Documents/ML/wide_kalshi_test.csv", index=False)

In [ ]:
n_tickers =  kalshi_w_markets.groupby('timestamp').size().reset_index(name='n_obs')

small_ts = n_tickers[n_tickers['n_obs'] < 5]
print("timestamps with <5 obs:", len(small_ts))

n_tickers['few_obs_flag'] = n_tickers['n_obs'] < 5
kalshi_w_markets = kalshi_w_markets.merge(n_tickers, on='timestamp')
few_obs_rows = kalshi_w_markets[kalshi_w_markets['n_obs'] < 5]
print("rows in small groups:", len(few_obs_rows))

few_obs_rows.to_csv("/Users/marcotaylhardat/Documents/ML/few_obs_rows.csv", index=False)

timestamps with <5 obs: 1140
rows in small groups: 3855


In [44]:
print(kalshi_w_markets.head())

                         ticker  end_period_ts  volume  open_interest  open  \
206958  KXFEDDECISION-26APR-H26     1765447200       0              0   NaN   
210085  KXFEDDECISION-26APR-H25     1765447200       0              0   NaN   
213236   KXFEDDECISION-26APR-H0     1765447200       0            787   NaN   
216907  KXFEDDECISION-26APR-C26     1765447200       0            611   NaN   
220507  KXFEDDECISION-26APR-C25     1765447200       0            420   NaN   

        high  low  close  mean                 timestamp ticker_suffix  \
206958   NaN  NaN    NaN   NaN 2025-12-11 10:00:00+00:00           H26   
210085   NaN  NaN    NaN   NaN 2025-12-11 10:00:00+00:00           H25   
213236   NaN  NaN    NaN   NaN 2025-12-11 10:00:00+00:00            H0   
216907   NaN  NaN    NaN   NaN 2025-12-11 10:00:00+00:00           C26   
220507   NaN  NaN    NaN   NaN 2025-12-11 10:00:00+00:00           C25   

       ticker_middle                close_time                 open_time  \
2069